# D8_baby_sound — 무라벨(라벨링 없이) 획득

AudioSet 527클래스로 파인튜닝된 AST(MIT/ast-finetuned-audioset-10-10-0.4593)를 제로샷 태거로 사용해 수동 라벨 없이 아기 소리 존재를 점수화한다. manifest의 sound_iv 구간만 잘라(±0.25s 패딩, 병합, 최대 8구간) AST에 배치로 넣고 아기 관련 클래스(Baby cry/Babbling/Baby laughter/Crying, sobbing/Whimper)의 sigmoid 점수 최댓값을 p_baby로 집계하며, sound_frac==0인 무음 클립은 모델 호출 없이 0으로 처리해 T4 비용을 줄인다. 성인 발화 점수 p_speech를 함께 기록해 어른 말소리와의 혼동을 검증 셀에서 확인하고, 임계값(기본 0.30)은 히스토그램을 보고 조정한다. AST는 PANNs(mAP 0.439)보다 AudioSet 태깅 성능이 높고(mAP 0.485) Colab 기본 설치된 transformers 한 줄로 로드된다.

In [ ]:
# 설치 셀 — 처음 실행 후 protobuf/numpy 경고가 나오면: 런타임 → 세션 다시 시작 → 이어서 실행
!pip install -q transformers soundfile av decord

In [ ]:
# 공통 설정 — Drive 마운트 + 경로 + manifest
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd

BASE = Path('/content/drive/MyDrive/BabyMon/dataset')   # ← 업로드 위치
CLIPS = None
for cand in (BASE/'clips', BASE/'resized_2', BASE):
    if cand.is_dir() and next(cand.glob('*.mp4'), None):
        CLIPS = cand; break
assert CLIPS, 'mp4 폴더 없음 (clips/ 또는 resized_2/)'
assert (BASE/'manifest.csv').exists(), 'manifest.csv 를 BASE 에 업로드하세요'
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
man = pd.read_csv(BASE/'manifest.csv')
print('clips:', CLIPS, len(list(CLIPS.glob("*.mp4"))), '개 / manifest:', len(man))

## D8_baby_sound — 아기 소리 존재 (무라벨 자동 라벨링)

**방법**: AudioSet 527클래스로 파인튜닝된 **AST**(Audio Spectrogram Transformer, `MIT/ast-finetuned-audioset-10-10-0.4593`)를 제로샷 태거로 사용한다.

1. `sound_frac == 0` 인 클립은 모델 호출 없이 음성(0) 처리 — GPU 비용 절약.
2. 소리가 있으면 manifest의 `sound_iv` 구간을 ±0.25초 패딩/병합해 최대 8구간을 잘라 AST에 배치 입력 (구간 단위 태깅이 8초 전체 입력보다 짧은 아기 소리에 민감).
3. 아기 관련 클래스(**Baby cry, infant cry / Babbling / Baby laughter / Crying, sobbing / Whimper**)의 sigmoid 점수를 구간·클래스에 걸쳐 max 집계 → `p_baby`. 성인 발화 계열 `p_speech` 도 함께 기록해 혼동 검증.
4. 임계값 기본 0.30 — 검증 셀 히스토그램을 보고 조정.

8kHz 원음을 16kHz로 업샘플해도(4kHz 이상 대역 없음) 울음 기본주파수(300–600Hz)와 하모닉은 보존되므로 실용상 문제 없다.

**출처**
- AST (Interspeech 2021): https://github.com/YuanGongND/ast — mAP 0.485, PANNs(0.439)보다 우수
- PANNs (TASLP 2020): https://arxiv.org/abs/1912.10211 — AudioSet 태거 제로샷 접근의 근거
- ICSD 유아 울음 검출 데이터셋 (arXiv 2024): https://arxiv.org/html/2408.10561v1
- 약지도 아기 울음 검출 (arXiv 2023): https://arxiv.org/html/2304.10001

In [ ]:
# 모델 초기화: AST AudioSet 태거 (transformers)
import torch, numpy as np
from transformers import AutoFeatureExtractor, ASTForAudioClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'MIT/ast-finetuned-audioset-10-10-0.4593'  # AudioSet 527클래스
extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = ASTForAudioClassification.from_pretrained(MODEL_ID).to(DEVICE).eval()
id2label = model.config.id2label
label2id = {v: k for k, v in id2label.items()}

def find_ids(names):
    # 정확 일치 우선, 없으면 부분 일치 (체크포인트 버전별 라벨 표기 차이 대비)
    ids = []
    for n in names:
        if n in label2id:
            ids.append(label2id[n])
        else:
            hit = [i for i, l in id2label.items() if n.lower() in l.lower()]
            ids += hit[:1]
    return sorted(set(ids))

BABY_IDS = find_ids(['Baby cry, infant cry', 'Babbling', 'Baby laughter',
                     'Crying, sobbing', 'Whimper'])
SPEECH_IDS = find_ids(['Speech', 'Child speech, kid speaking',
                       'Male speech, man speaking', 'Female speech, woman speaking'])

def ckey(i):  # 클래스별 결과 컬럼명 (예: p_baby_cry)
    return 'p_' + id2label[i].split(',')[0].lower().replace(' ', '_')

print('baby  :', [id2label[i] for i in BABY_IDS])
print('speech:', [id2label[i] for i in SPEECH_IDS])

In [ ]:
# 클립 1개 처리: sound_iv 구간만 잘라 AST 태깅 -> dict 반환
import json, os, subprocess, tempfile
import soundfile as sf

SR = 16000  # AST 입력 샘플레이트 (8kHz AAC -> ffmpeg 업샘플)

def load_audio(path, sr=SR):
    fd, tmp = tempfile.mkstemp(suffix='.wav'); os.close(fd)
    try:
        subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', str(path),
                        '-ac', '1', '-ar', str(sr), tmp], check=True)
        wav, _ = sf.read(tmp, dtype='float32')
    finally:
        os.remove(tmp)
    return wav

def sound_segments(iv_json, dur):
    """sound_iv 구간을 ±0.25s 패딩, 최소 1s 확보, 겹치면 병합. 없으면 전체."""
    try:
        ivs = json.loads(iv_json) if isinstance(iv_json, str) else list(iv_json)
    except Exception:
        ivs = []
    segs = []
    for s, e in ivs:
        s, e = max(0., float(s) - .25), min(dur, float(e) + .25)
        if e - s < 1.0:
            c = (s + e) / 2
            s, e = max(0., c - .5), min(dur, c + .5)
        if segs and s <= segs[-1][1]:
            segs[-1][1] = max(segs[-1][1], e)
        else:
            segs.append([s, e])
    return segs if segs else [[0., dur]]

@torch.no_grad()
def process_clip(row):
    out = {'file': row['file'], 'sound_frac': float(row['sound_frac'])}
    if out['sound_frac'] <= 0.0:              # 무음 클립: 모델 생략
        out.update(n_seg=0, p_baby=0.0, p_speech=0.0)
        out.update({ckey(i): 0.0 for i in BABY_IDS})
        return out
    wav = load_audio(CLIPS / row['file'])
    if len(wav) < SR // 5:
        raise ValueError('no/short audio stream')
    dur = len(wav) / SR
    segs = sound_segments(row.get('sound_iv'), dur)[:8]  # 최대 8구간
    chunks = [wav[int(s * SR):int(e * SR)] for s, e in segs]
    chunks = [c for c in chunks if len(c) > SR // 5] or [wav]
    # extractor 는 ndarray 리스트를 받아 1024프레임으로 패딩/절단 (transformers AST)
    feat = extractor(chunks, sampling_rate=SR, return_tensors='pt')
    logits = model(feat.input_values.to(DEVICE)).logits
    probs = torch.sigmoid(logits).cpu().numpy()           # (n_seg, 527)
    out['n_seg'] = len(chunks)
    out['p_baby'] = float(probs[:, BABY_IDS].max())       # 구간x클래스 max
    out['p_speech'] = float(probs[:, SPEECH_IDS].max())
    for i in BABY_IDS:
        out[ckey(i)] = float(probs[:, i].max())
    return out

In [ ]:
# 배치 루프: 증분 저장 (이미 처리한 file 건너뜀), 50개마다 진행 출력
OUT_CSV = BASE / 'auto_D8_baby_sound.csv'
N_CLIPS = 1000  # 표본 크기 (전량 처리 시 len(man) 으로)
COLS = (['file', 'sound_frac', 'n_seg', 'p_baby', 'p_speech']
        + [ckey(i) for i in BABY_IDS] + ['error'])

done = set(pd.read_csv(OUT_CSV)['file']) if OUT_CSV.exists() else set()
sample = man.sample(n=min(N_CLIPS, len(man)), random_state=0)
todo = sample[~sample['file'].isin(done)]
print(f'대상 {len(sample)}개 중 신규 {len(todo)}개 (기존 완료 {len(done)}개)')

def flush(buf):
    if buf:
        pd.DataFrame(buf).reindex(columns=COLS).to_csv(
            OUT_CSV, mode='a', header=not OUT_CSV.exists(), index=False)
    return []

buf, n_done = [], 0
for _, row in todo.iterrows():
    try:
        buf.append(process_clip(row))
    except Exception as e:                    # 손상 파일 등은 기록 후 계속
        buf.append({'file': row['file'], 'error': str(e)[:100]})
    n_done += 1
    if n_done % 50 == 0:
        buf = flush(buf)
        print(f'{n_done}/{len(todo)} 처리 완료')
buf = flush(buf)
print(f'종료: 신규 {n_done}개 -> {OUT_CSV}')

In [ ]:
# 검증: p_baby 분포 + 극단 사례 확인 (임계값 TH 는 분포 보고 조정)
import matplotlib.pyplot as plt
from IPython.display import Audio, display

df = pd.read_csv(OUT_CSV)
if 'error' in df.columns:
    print('에러 클립:', df['error'].notna().sum())
    df = df[df['error'].isna() & df['p_baby'].notna()].copy()
TH = 0.30
df['baby_sound'] = (df['p_baby'] >= TH).astype(int)
print(f"{len(df)}개 | 임계값 {TH} 기준 양성비율 {df['baby_sound'].mean():.3f}")

BLUE = '#4269D0'  # 축/제목은 영어 (Colab 한글 폰트 미설정 대비)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(df['p_baby'], bins=40, color=BLUE)
ax[0].axvline(TH, color='#9498A0', lw=1, ls='--')
ax[0].set_xlabel('p_baby'); ax[0].set_ylabel('clips')
ax[0].set_title('p_baby distribution')
ax[1].scatter(df['sound_frac'], df['p_baby'], s=6, alpha=.35,
              color=BLUE, edgecolors='none')
ax[1].set_xlabel('sound_frac'); ax[1].set_ylabel('p_baby')
ax[1].set_title('p_baby vs sound activity')
for a in ax:
    a.grid(alpha=.25)
    a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

cols = ['file', 'p_baby', 'p_speech', 'sound_frac']
print('--- p_baby 상위 5 (아기 소리 확실 사례) ---')
print(df.nlargest(5, 'p_baby')[cols].to_string(index=False))
print('--- 소리 많은데 p_baby 낮음 (어른 말/생활소음 추정) ---')
print(df[df['sound_frac'] > .3].nsmallest(5, 'p_baby')[cols].to_string(index=False))
for f in df.nlargest(3, 'p_baby')['file']:    # 직접 청취 검증
    try:
        display(f, Audio(load_audio(CLIPS / f), rate=SR))
    except Exception as e:
        print(f, 'audio 재생 실패:', e)